# 第 5 周练习：公司知识库 RAG 助理

## 练习目标（理念）

在本课第 5 周的知识库上，搭建一个**本地 RAG（Retrieval-Augmented Generation）助手**：

1. 读取知识库里的 `.md` / `.txt` 文件  
2. 把长文**分块（chunking）**  
3. 用 Hugging Face 嵌入模型把块写成向量，存进 **Chroma**  
4. 按问题检索相关段落，再交给聊天模型作答，并要求给出**引文（citations）**

## 和本课 Week 5 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Document loader | `TextLoader` + `rglob` 扫知识库 |
| Text splitter | `RecursiveCharacterTextSplitter` |
| Embeddings | `HuggingFaceEmbeddings`（本地模型 `all-MiniLM-L6-v2`） |
| Vector store | `Chroma.from_documents` |
| RAG 问答 | `SYSTEM_PROMPT` + `ask_kb` |

## 怎么跑

1. 按需安装依赖（下一格的 `pip` 注释）  
2. 准备 `.env`：`OPENAI_API_KEY`（此笔记本走 OpenRouter 的 `base_url`）  
3. 把 `KB_PATH` 改成你本机的 `week5/knowledge-base` 路径  
4. 从上到下依次运行；最后一格是示例提问  

> 注意：原文里检索器 `retriever` / `format_context` 若未在前面单元格定义，需要你对照课程补全后再调用 `ask_kb`。


In [ ]:
# ========== 可选依赖安装（默认注释掉，避免每次自动 pip）==========
# 如果需要，安装依赖项：LangChain 生态 + Chroma + sentence-transformers（嵌入后端）
# !pip -q install langchain langchain-community langchain-text-splitters chromadb sentence-transformers


In [23]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 标准库 os：读环境变量（Environment Variables），例如 API Key
import os
# pathlib.Path：用面向对象方式拼路径、递归扫文件
from pathlib import Path
# TextLoader：把单个文本文件加载成 LangChain Document
from langchain_community.document_loaders import TextLoader
# RecursiveCharacterTextSplitter：按字符递归切块，兼顾重叠（overlap）
from langchain_text_splitters import RecursiveCharacterTextSplitter
# HuggingFaceEmbeddings：本地跑句向量模型，把文本变成 embedding
from langchain_community.embeddings import HuggingFaceEmbeddings
# Chroma：轻量向量库，把文档块 + 向量持久化到本地目录
from langchain_community.vectorstores import Chroma
# OpenAI 客户端：这里通过 OpenRouter 兼容接口调用聊天模型
from openai import OpenAI
# load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv


In [24]:
# ========== 环境变量 + OpenRouter 客户端 ==========

# 加载 .env；override=True 表示用文件里的值覆盖已有环境变量
load_dotenv(override=True)
# 从环境变量读取 OpenAI 兼容密钥（OpenRouter 也常用这个名字）
api_key = os.getenv('OPENAI_API_KEY')

# 缺密钥：提示去 .env 里补（英文文案保持原样，可能被程序/同学对照）
if not api_key:
    print('No API key found. Please add OPENAI_API_KEY to your .env file.')
# 密钥两端有空格：常见复制粘贴坑
elif api_key.strip() != api_key:
    print('API key has leading/trailing whitespace. Please remove it.')
# 看起来正常
else:
    print('API key looks good!')

# 构造 OpenAI 客户端：把请求打到 OpenRouter 的兼容 API
openai = OpenAI(
    api_key=api_key,
    base_url='https://openrouter.ai/api/v1',
)
# 选用的聊天模型 id（必须保持原字符串，勿改）
MODEL = 'gpt-4.1-mini'


API key looks good!


In [25]:
# ========== 加载知识库文本文件 ==========

# 知识库根目录（作者本机绝对路径；换成你自己的 week5/knowledge-base）
KB_PATH = Path('/Users/andela/projects/llm_engineering/week5/knowledge-base')

# 递归收集根目录下所有 .md / .txt，加载为 Document 列表
def load_kb_text_files(root: Path):
    # docs：累积所有加载到的文档
    docs = []
    # 两种扩展名都扫一遍
    for ext in ('*.md', '*.txt'):
        # rglob：递归匹配相对模式
        for path in root.rglob(ext):
            # TextLoader(str(path)).load()：读文件并转成 Document 列表，再 extend 进去
            docs.extend(TextLoader(str(path)).load())
    return docs

# 真正加载知识库
raw_docs = load_kb_text_files(KB_PATH)
# 打印加载了多少篇文档，便于确认路径是否正确
print(f'Loaded {len(raw_docs)} documents')
# 一篇都没有：路径或扩展名不对，直接抛错中断
if len(raw_docs) == 0:
    raise ValueError('No documents found. Check KB_PATH and file extensions.')


Loaded 76 documents


In [26]:
# ========== 分块（chunking）：长文切成可检索的小段 ==========

# chunk_size=900：每块大约 900 字符；chunk_overlap=120：块与块重叠，减少边界信息丢失
splitter = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=120)
# 对刚加载的原始文档做切分，得到 chunks
chunks = splitter.split_documents(raw_docs)
# 打印块数量，粗看切分规模
print(f'Chunks: {len(chunks)}')


Chunks: 450


In [27]:
# ========== 嵌入模型 + Chroma 向量库（本地持久化）==========

# 句向量模型名：轻量、适合本地 CPU 跑
EMBED_MODEL = 'all-MiniLM-L6-v2'
# 用 Hugging Face 封装创建嵌入器
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
# 把所有 chunks 写入 Chroma；persist_directory 指定本地落盘目录
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory='week5_kb_chroma'
)
# 距离分数越小通常越相似；后面若做过滤可调这个阈值（原文定义了常量）
MAX_DISTANCE = 0.8


In [28]:
# ========== 系统提示：约束「只根据上下文答 + 必须给引文」==========
# 下面整段英文 prompt 字符串是可执行行为的一部分，必须原样保留，不要翻译

SYSTEM_PROMPT = '''
You are a company knowledge base assistant.
Answer using ONLY the provided context.
If the context is insufficient, say so clearly.
Return markdown with:
# 回答
# 引文
Citations must be bullet points with file paths.
'''


In [29]:
# ========== 问答函数：检索 → 拼上下文 → 调聊天模型 ==========

# ask_kb：给定自然语言问题，走完整 RAG 回答链路
def ask_kb(question: str):
    # 较新的 LangChain 检索器使用 invoke()（需事先有名为 retriever 的对象）
    docs = retriever.invoke(question)
    # 把检索到的 Document 列表格式化成一段给模型看的 Context 文本
    context = format_context(docs)
    # 组装 Chat Completions 的 messages：system 定规矩，user 放问题+上下文
    messages = [
        # system：只根据上下文答，并要求 markdown 引文
        {'role': 'system', 'content': SYSTEM_PROMPT},
        # user：问题原文 + 检索上下文（Question/Context 英文前缀保持原样）
        {'role': 'user', 'content': 'Question: ' + question + '\n\nContext:\n' + context}
    ]
    # 调用 OpenRouter 上的聊天模型
    response = openai.chat.completions.create(
        # 模型 id 来自前面的 MODEL 常量
        model=MODEL,
        # 刚组装好的 messages
        messages=messages,
        # 限制回答长度，避免过长输出
        max_tokens=600,
    )
    # 取出第一条 choice 的文本内容返回
    return response.choices[0].message.content


In [30]:
# ========== 示例提问：跑通 RAG 问答链路 ==========

# 问休假/假期政策（英文问题字符串保持原样）
print(ask_kb('What is the company policy on PTO and holidays?'))
# 空行分隔两次回答，方便阅读
print()
# 问旗舰产品关键特性
print(ask_kb('What are the key features of the flagship product?'))


## Answer
The provided context does not include any information regarding the company policy on PTO (Paid Time Off) and holidays.

## Citations
- /Users/andela/projects/llm_engineering/week5/knowledge-base/contracts/Contract with Metropolitan Life Group for Lifellm.md
- /Users/andela/projects/llm_engineering/week5/knowledge-base/contracts/Contract with GlobalRe Partners for Rellm.md

## Answer

The key features of the flagship product include:

1. **Account Management**:
   - Named Customer Success Manager with weekly check-ins for the first 90 days, then bi-weekly.
   - Quarterly executive business reviews with metrics analysis.
   - Annual strategic planning session.
   - Direct escalation path to the VP of Customer Success.

2. **Integration Services**:
   - Integration with core systems such as Guidewire ClaimCenter, Salesforce CRM, DocuSign for settlement documents, and payment processing systems (CheckFree, AvidXchange).
   - Custom API development (with up to 100 hours included)